# CEO Cost & Margin Dashboard — D2C Fashion

**⚠️ ALL DATA IN THIS NOTEBOOK IS SYNTHETIC.** It is generated by a seeded
random data generator to look like a plausible small D2C fashion brand. It
is not real sales, marketing, or inventory data, and no business conclusion
should be drawn from the specific numbers — only from the *methodology*.

## What this notebook is

A two-layer analytics engine for a goods-selling business's cost and margin
performance, built so a CEO can see revenue, margin, working capital,
returns, acquisition efficiency, and inventory health in one place — with
every number traceable to a plain-language formula.

**Layer 1 — Universal Core**: KPIs that apply to *any* goods business
(revenue, gross margin, contribution margin, COGS/opex breakdown,
inventory turns, working-capital cycle, SKU concentration).

**Layer 2 — Fashion Module**: KPIs specific to D2C fashion (returns rate,
CAC/ROAS/MER, LTV:CAC, AOV, repeat rate, cohort retention, sell-through,
weeks-of-cover, markdown/dead-stock %), defined against the **same** core
engine via a config file rather than by editing the engine itself. Swapping
industries (e.g. to FMCG or auto parts) is meant to mean *writing a new
config*, not touching Layer 1's calculation code.

## No black boxes

Every KPI's formula and the business question it answers is written out in
a markdown cell before it's used. If a line of code in this notebook can't
be explained in plain language in an interview, it doesn't belong here.

## Build stages (this notebook is built incrementally)

1. **Data generator + swappable loader** ← this notebook, current stage
2. Cleaning + Universal Core KPIs
3. Fashion industry module (returns, CAC, cohorts, sell-through)
4. Dashboard / visualizations
5. Margin-risk alert model
6. Demand / inventory forecast model

---
## Stage 1 — Synthetic data generator and the swappable column-mapping loader

**Goal of this stage:** produce a realistic order-level D2C fashion dataset
and prove that swapping in a real CSV later requires editing only a
column-mapping dictionary — never the analysis code.

### Setup — installs (Colab) and imports

In [1]:
# Colab: uncomment the line below on first run in a fresh Colab environment.
# !pip install -q pandas numpy matplotlib plotly scikit-learn

import sys
sys.path.append("..")  # so `from src import ...` resolves when running from notebooks/

import pandas as pd
import numpy as np

from src import schema
from src import data_generator as dg
from src import data_loader as dl

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

### Why three tables, not one giant spreadsheet

A real D2C business's data doesn't naturally live in a single flat table,
and forcing it into one would hide real structure:

| Table | Grain | Why it's separate |
|---|---|---|
| `orders` | one row per **order line item** | The core fact table: revenue, cost, returns, and customer identity all live here. |
| `marketing_spend` | one row per **(month, channel)** | Ad spend is a channel-level input (an ad platform invoice) — it isn't a property of one order, so attaching a "spend" column to every order line would double- or triple-count it the moment an order has multiple line items. |
| `inventory_snapshots` | one row per **(month, SKU)** | Turns, days-of-inventory, and sell-through all need what *wasn't* sold (opening/closing stock), which the orders table alone can never tell you. |

This mirrors how the data actually arrives from a real stack (Shopify/POS
export, ad-platform report, WMS export) — three separate files that get
joined for analysis, not reconciled from a single source.

### Generating the synthetic dataset

`src/data_generator.py` builds all three tables from a single seeded
`numpy.random.Generator` (default seed 42), so the dataset is 100%
reproducible — rerunning this cell (or the whole notebook) on any machine
produces byte-identical output.

Key generation choices, made explicit here so nothing downstream is a
surprise:
- **12 months** of order-line data (Jul 2025 – Jun 2026), 6 product
  categories, ~150 SKUs, an order-volume ramp (the brand is growing) and two
  seasonal sale-driven demand spikes.
- **Return rates are category-specific and deliberately high for
  fit-sensitive categories** (Dresses ~38%, Tops ~32%, Bottoms ~28%),
  landing in the 20–40% range real fashion brands see, specifically so the
  margin impact of returns is visible rather than a rounding error.
- **Customers arrive over time and a share of them re-order** later in the
  year, which is what makes repeat-purchase-rate and cohort retention
  (Stage 3) meaningful rather than trivially zero.
- **Marketing spend per channel roughly tracks the orders that channel is
  credited with**, plus noise and a channel-specific efficiency factor —
  so CAC/ROAS differ meaningfully by channel instead of being flat by
  construction.
- **Inventory purchasing is deliberately imperfect for a subset of SKUs**
  (some over-bought, some under-bought) so later stages have genuine
  overstock/stockout cases to detect.

Full assumptions are documented in the `data_generator.py` module
docstring.

In [2]:
data = dg.generate_all(seed=42)
orders_df = data["orders"]
marketing_df = data["marketing_spend"]
inventory_df = data["inventory_snapshots"]

print("SYNTHETIC DATA — generated with seed=42, fully reproducible\n")
for name, df in data.items():
    print(f"{name:22s} shape={df.shape}")

SYNTHETIC DATA — generated with seed=42, fully reproducible

orders                 shape=(25930, 15)
marketing_spend        shape=(60, 3)
inventory_snapshots    shape=(1651, 6)


### Orders table — schema and sample

In [3]:
print("orders.csv column dtypes:\n")
print(orders_df.dtypes)
orders_df.head(5)

orders.csv column dtypes:

order_id                          str
order_line_id                     str
order_date             datetime64[us]
customer_id                       str
sku_id                            str
category                          str
size                              str
quantity                        int64
unit_price                    float64
unit_cogs                     float64
shipping_cost                 float64
payment_gateway_fee           float64
marketing_channel                 str
is_return                        bool
return_date            datetime64[us]
dtype: object


,order_id,order_line_id,order_date,customer_id,sku_id,category,size,quantity,unit_price,unit_cogs,shipping_cost,payment_gateway_fee,marketing_channel,is_return,return_date
0,ORD-000001,ORD-000001-1,2025-07-01,CUST-00001,BOT-025,Bottoms,M,1,1859.0,754.26,49.15,41.04,Organic/Email,True,2025-07-10
1,ORD-000015,ORD-000015-1,2025-07-01,CUST-00014,ACC-009,Accessories,One Size,1,1119.0,390.44,62.30,25.50,Affiliate,False,NaT
2,ORD-000016,ORD-000016-1,2025-07-01,CUST-00015,TOP-023,Tops,M,1,879.0,300.27,77.87,20.46,Instagram Ads,False,NaT
3,ORD-000017,ORD-000017-1,2025-07-01,CUST-00007,OUT-007,Outerwear,M,1,2649.0,1026.21,72.83,57.63,Organic/Email,False,NaT
4,ORD-000018,ORD-000018-1,2025-07-01,CUST-00016,FOO-016,Footwear,11,2,3639.0,1557.89,91.71,154.84,Influencer,False,NaT


**Column-by-column explanation of `orders`** (one row = one order line item):

| Column | Meaning |
|---|---|
| `order_id` | Groups line items placed in the same checkout — an order with 2 SKUs is 2 rows sharing this value. |
| `order_line_id` | Unique key for this row (`order_id` + line number). |
| `order_date` | Date the order was placed. |
| `customer_id` | Identifies the buyer; the same value recurs across a customer's repeat orders — this is what makes repeat-rate and cohort analysis possible. |
| `sku_id` | Product+variant identifier (e.g. `DRE-014`). |
| `category` | Product category — Tops, Bottoms, Dresses, Outerwear, Footwear, or Accessories. |
| `size` | Size variant of the SKU. |
| `quantity` | Units on this line (mostly 1, occasionally 2). |
| `unit_price` | Realized selling price per unit, **after** any discount — this is what actually hit revenue, not the list price. |
| `unit_cogs` | Cost of goods per unit for this SKU (manufacturing/sourcing cost). |
| `shipping_cost` | Fulfillment/shipping cost allocated to this line. |
| `payment_gateway_fee` | Payment processor fee allocated to this line (~2.1% + a small fixed fee, typical of Indian payment gateways). |
| `marketing_channel` | The channel credited with driving the *parent order* (Instagram Ads, Google Ads, Influencer, Affiliate, or Organic/Email). |
| `is_return` | `True` if this specific line item was returned. |
| `return_date` | Date of the return; `NaT` (missing) if never returned. |

### Marketing spend table — schema and sample

In [4]:
print("marketing_spend.csv column dtypes:\n")
print(marketing_df.dtypes)
marketing_df.head(5)

marketing_spend.csv column dtypes:

month          str
channel        str
spend      float64
dtype: object


,month,channel,spend
0,2025-07,Affiliate,22688.01
1,2025-07,Google Ads,96084.07
2,2025-07,Influencer,62745.79
3,2025-07,Instagram Ads,83227.15
4,2025-07,Organic/Email,0.00


**Column-by-column explanation of `marketing_spend`** (one row = one
channel's total spend in one month):

| Column | Meaning |
|---|---|
| `month` | Calendar month, `YYYY-MM`. |
| `channel` | Marketing channel (matches `orders.marketing_channel`). |
| `spend` | Total spend on that channel that month. `Organic/Email` is always 0 — it represents word-of-mouth, direct, and retention email, not a paid acquisition channel. |

This table is joined to `orders` (aggregated by month + channel) when
computing CAC/ROAS in Stage 3 — it is never merged row-for-row onto order
lines, because that would silently duplicate spend across every line item
of a multi-item order.

### Inventory snapshots table — schema and sample

In [5]:
print("inventory_snapshots.csv column dtypes:\n")
print(inventory_df.dtypes)
inventory_df.head(5)

inventory_snapshots.csv column dtypes:

sku_id                   str
month                    str
beginning_inventory    int64
units_received         int64
units_sold             int64
ending_inventory       int64
dtype: object


,sku_id,month,beginning_inventory,units_received,units_sold,ending_inventory
0,ACC-001,2025-09,26,27,11,42
1,ACC-001,2025-10,42,0,6,36
2,ACC-001,2025-11,36,0,28,8
3,ACC-001,2025-12,8,25,17,16
4,ACC-001,2026-01,16,25,20,21


**Column-by-column explanation of `inventory_snapshots`** (one row = one
SKU's stock position in one month):

| Column | Meaning |
|---|---|
| `sku_id` | Matches `orders.sku_id`. |
| `month` | Calendar month, `YYYY-MM`. |
| `beginning_inventory` | Units on hand at the start of the month. |
| `units_received` | Units restocked during the month. |
| `units_sold` | Units sold during the month (gross, before returns) — reconciles with `orders` grouped by SKU and month. |
| `ending_inventory` | `beginning_inventory + units_received - units_sold`, floored at 0. Next month's `beginning_inventory` always equals this value — verified in the test suite. |

### Quick sanity check: does this data look like a real fashion brand?

In [6]:
total_revenue = (orders_df["unit_price"] * orders_df["quantity"]).sum()
print(f"Date range        : {orders_df['order_date'].min().date()} to {orders_df['order_date'].max().date()}")
print(f"Unique orders      : {orders_df['order_id'].nunique():,}")
print(f"Unique customers   : {orders_df['customer_id'].nunique():,}")
print(f"Unique SKUs        : {orders_df['sku_id'].nunique()}")
print(f"Order line items   : {len(orders_df):,}")
print(f"Gross revenue (pre-return, INR): {total_revenue:,.0f}")
print()
print("Return rate by category (target: fit-sensitive categories land in the realistic 20-40% fashion range):")
print(orders_df.groupby("category")["is_return"].mean().sort_values(ascending=False).round(3))

Date range        : 2025-07-01 to 2026-06-30
Unique orders      : 18,607
Unique customers   : 12,002
Unique SKUs        : 150
Order line items   : 25,930
Gross revenue (pre-return, INR): 51,692,051

Return rate by category (target: fit-sensitive categories land in the realistic 20-40% fashion range):
category
Dresses        0.377
Tops           0.327
Bottoms        0.285
Footwear       0.212
Outerwear      0.171
Accessories    0.088
Name: is_return, dtype: float64


Dresses, Tops, and Bottoms — the fit-sensitive categories — land in the
28–38% return-rate range, Accessories sits under 10%. This spread is by
design (see the generator's category return-rate table) and is what will
make the returns-driven margin impact visible in Stage 3, rather than a
number too small to matter.

### The swappable loader — proving a real CSV export could replace this data with zero analysis-code changes

In [7]:
import os

# Save the generated tables exactly as a real export would arrive: plain CSVs.
os.makedirs("../data/synthetic", exist_ok=True)
dg.save_all(data, "../data/synthetic")
print("Saved:", os.listdir("../data/synthetic"))

Saved: ['inventory_snapshots.csv', 'marketing_spend.csv', 'orders.csv']


`src/data_loader.py` reads these CSVs and renames columns using a mapping
dict (`schema.DEFAULT_ORDERS_MAPPING`, etc.) — by default an identity
mapping, since the synthetic data already uses canonical column names. To
prove this is a real swap point (not just a pass-through), the cell below
simulates a real export with **different column headers** — `"Order No"`
instead of `order_id`, `"Selling Price"` instead of `unit_price` — and
loads it by editing only a mapping dict, changing no code in
`data_loader.py` or any calculation module.

In [8]:
# Simulate a real export with different headers than our canonical schema.
fake_export = pd.read_csv("../data/synthetic/orders.csv")
fake_export = fake_export.rename(columns={"order_id": "Order No", "unit_price": "Selling Price"})
fake_export.to_csv("../data/synthetic/_demo_real_export.csv", index=False)

# Build a custom mapping: copy the default, then repoint the two changed headers.
custom_mapping = dict(schema.DEFAULT_ORDERS_MAPPING)
del custom_mapping["order_id"]
del custom_mapping["unit_price"]
custom_mapping["Order No"] = schema.ORDER_ID
custom_mapping["Selling Price"] = schema.UNIT_PRICE

swapped_df = dl.load_orders("../data/synthetic/_demo_real_export.csv", mapping=custom_mapping)
print("Loaded a differently-headed file using only a mapping-dict edit.")
print("Resulting columns match the canonical schema:", list(swapped_df.columns) == schema.ORDERS_SCHEMA)
swapped_df.head(3)

Loaded a differently-headed file using only a mapping-dict edit.
Resulting columns match the canonical schema: True


,order_id,order_line_id,order_date,customer_id,sku_id,category,size,quantity,unit_price,unit_cogs,shipping_cost,payment_gateway_fee,marketing_channel,is_return,return_date
0,ORD-000001,ORD-000001-1,2025-07-01,CUST-00001,BOT-025,Bottoms,M,1,1859.0,754.26,49.15,41.04,Organic/Email,True,2025-07-10
1,ORD-000015,ORD-000015-1,2025-07-01,CUST-00014,ACC-009,Accessories,One Size,1,1119.0,390.44,62.30,25.50,Affiliate,False,NaT
2,ORD-000016,ORD-000016-1,2025-07-01,CUST-00015,TOP-023,Tops,M,1,879.0,300.27,77.87,20.46,Instagram Ads,False,NaT


In [9]:
os.remove("../data/synthetic/_demo_real_export.csv")  # clean up the demo file

If a real export is missing a column the engine needs, the loader raises
immediately with the missing column's name (`schema.validate_columns`),
rather than letting a KPI compute silently on absent data. This is checked
in `tests/test_stage1_data.py::test_loader_raises_on_missing_column`.

---
## Stage 1 summary

Built a fully synthetic, seeded, reproducible 12-month D2C fashion dataset
across three tables — `orders` (order-line grain, 25,930 lines / ~18,600
orders / 12,002 customers / 150 SKUs), `marketing_spend` (channel × month),
and `inventory_snapshots` (SKU × month) — with category-specific return
rates realistically landing in the 20–40% fashion range (Dresses ~38%,
Tops ~32%) so their margin impact will be visible later. Every column is
documented above. The data-loading layer (`src/data_loader.py` +
`src/schema.py`) was proven swappable: a differently-headed CSV loads
correctly by editing only a column-mapping dictionary, with no changes to
loader or analysis code, and a missing required column fails loudly instead
of silently. 9 automated tests cover schema shape, reproducibility,
inventory-chain consistency, and the loader swap/failure paths (all
passing).

**Stopping here for review before Stage 2 (cleaning + Universal Core KPIs).**